In [1]:
import nltk
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("../input/nlp-getting-started/train.csv")

In [3]:
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [4]:
df.shape

(7613, 5)

In [5]:
df.isnull().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [6]:
df.drop('keyword', axis=1,inplace=True)
df.drop('location', axis=1, inplace=True)

In [7]:
df.target.nunique() #binary classifier

2

In [8]:
df.target.value_counts()

0    4342
1    3271
Name: target, dtype: int64

In [9]:
df['text'][100]

'.@NorwayMFA #Bahrain police had previously died in a road accident they were not killed by explosion https://t.co/gFJfgTodad'

In [10]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
import re
#df.text = df.text.str.lower()
df.text = df.text.str.lower()

In [12]:
from nltk import *
from nltk.stem import WordNetLemmatizer

In [13]:
class LemmaTokenizer(object):
    def __init__(self):
        self.wnl = WordNetLemmatizer()
    def __call__(self, doc):
        return [self.wnl.lemmatize(t) for t in word_tokenize(doc)]

In [14]:
tfidf = TfidfVectorizer(max_features = 5000, ngram_range=(1,1), tokenizer=LemmaTokenizer())

In [15]:
x = tfidf.fit_transform(df.text.to_list())

In [16]:
X = pd.DataFrame(x.toarray(), columns= tfidf.get_feature_names())

In [17]:
X.head()

,!,#,$,%,&,','','a,'armageddon,'avoiding,...,û÷good,û÷itûªs,û÷plot,û÷politics,«,å_,å¨,åç,åè,åê
0,0.0,0.145723,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.000000,0.0,0.0,0.0,0.124696,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.157123,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.235536,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
y = enc.fit_transform(df.target)

In [19]:
y

array([1, 1, 1, ..., 1, 1, 1])

In [20]:
enc.classes_

array([0, 1])

In [21]:
from sklearn.model_selection import train_test_split

In [22]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=33)

In [23]:
X_train.shape

(6090, 5000)

In [24]:
X_test.shape

(1523, 5000)

In [25]:
from sklearn.linear_model import LogisticRegression

In [26]:
clf1 = LogisticRegression()
clf1.fit(X_train,y_train)

LogisticRegression()

In [27]:
clf1.score(X_test,y_test)

0.7957977675640184

In [28]:
from sklearn.metrics import classification_report

In [29]:
print(classification_report(y_test,clf1.predict(X_test)))

              precision    recall  f1-score   support

           0       0.77      0.90      0.83       847
           1       0.84      0.67      0.74       676

    accuracy                           0.80      1523
   macro avg       0.81      0.78      0.79      1523
weighted avg       0.80      0.80      0.79      1523



In [30]:
test_data = pd.read_csv("../input/nlp-getting-started/test.csv")
test_data.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [31]:
test_data.shape

(3263, 4)

In [32]:
X_test1 = test_data.text.str.lower()
x1 = tfidf.fit_transform(df.text.to_list())
x1.shape

(7613, 5000)

In [33]:
X1 = pd.DataFrame(x1.toarray(), columns= tfidf.get_feature_names())
X1.shape

(7613, 5000)

In [34]:
pred = clf1.predict(X1)

In [35]:
p=pd.DataFrame()
p['id']=df['id']
p['target']=pred

In [36]:
p.to_csv('./Submission_athira.csv',index=False)